# 08.6b MusicGen 推理

MusicGen 代表 codec-token language model 路线。依赖和权重可用时，本 Notebook 会直接加载模型并生成音频；缺少依赖时会打印安装指引。


## 运行环境与安装

在 Jupyter 中选择 kernel：`Python 3.11 (chapter08-audiocraft)`。首次运行前先建立独立环境。mac/Linux 终端使用：

Hugging Face 当前 CLI 命令名为 `hf`。若 `hf --help` 不可用，可按官方文档先安装 standalone CLI；mac/Linux 使用 `curl -LsSf https://hf.co/cli/install.sh | bash`，Windows PowerShell 使用 `powershell -ExecutionPolicy ByPass -c "irm https://hf.co/cli/install.ps1 | iex"`。也可以把下面的 `hf download ...` 改成 `uvx hf download ...`。

```bash
cd CODE
python3.11 -m venv venv_ch08_audiocraft
source venv_ch08_audiocraft/bin/activate
python -m pip install --upgrade pip setuptools wheel
python -m pip install "numpy==1.26.4" "torch==2.1.0" "torchaudio==2.1.0" "torchvision==0.16.0" "torchtext==0.16.0"
python -m pip install --only-binary=:all: "av>=14"
python -m pip install einops "flashy>=0.0.1" "hydra-core>=1.1" hydra_colorlog julius num2words sentencepiece "spacy>=3.6.1" "huggingface_hub<1" tqdm "transformers==4.31.0" demucs librosa soundfile gradio torchmetrics encodec protobuf ipykernel ipywidgets
python -m pip install --no-deps audiocraft==1.3.0
python -m ipykernel install --user --name chapter08-audiocraft --display-name "Python 3.11 (chapter08-audiocraft)"
python -c "import sys; sys.path.insert(0, 'chapter08'); from model_runners.musicgen import install_xformers_compat_if_missing; install_xformers_compat_if_missing(); from audiocraft.models import MusicGen; print('MusicGen import ok')"
hf download facebook/musicgen-small --local-dir chapter08/models/facebook_musicgen_small
hf download t5-base --local-dir chapter08/models/t5_base
```

Windows PowerShell 使用：

```powershell
cd CODE
py -3.11 -m venv venv_ch08_audiocraft
.\venv_ch08_audiocraft\Scripts\Activate.ps1
python -m pip install --upgrade pip setuptools wheel
python -m pip install "numpy==1.26.4" "torch==2.1.0" "torchaudio==2.1.0" "torchvision==0.16.0" "torchtext==0.16.0"
python -m pip install --only-binary=:all: "av>=14"
python -m pip install einops "flashy>=0.0.1" "hydra-core>=1.1" hydra_colorlog julius num2words sentencepiece "spacy>=3.6.1" "huggingface_hub<1" tqdm "transformers==4.31.0" demucs librosa soundfile gradio torchmetrics encodec protobuf ipykernel ipywidgets
python -m pip install --no-deps audiocraft==1.3.0
python -m ipykernel install --user --name chapter08-audiocraft --display-name "Python 3.11 (chapter08-audiocraft)"
python -c "import sys; sys.path.insert(0, 'chapter08'); from model_runners.musicgen import install_xformers_compat_if_missing; install_xformers_compat_if_missing(); from audiocraft.models import MusicGen; print('MusicGen import ok')"
hf download facebook/musicgen-small --local-dir chapter08/models/facebook_musicgen_small
hf download t5-base --local-dir chapter08/models/t5_base
```

下载后本 Notebook 会自动优先使用 `chapter08/models/facebook_musicgen_small` 和 `chapter08/models/t5_base`。MusicGen 的 AudioCraft checkpoint 不包含 T5 文本编码器；若只下载 `facebook/musicgen-small`，AudioCraft 仍会尝试联网加载 `t5-base`。

设备选择仍按 `cuda -> mps -> cpu` 检测，但 AudioCraft 1.3.0 在 PyTorch 2.1 的 MPS autocast 路径会报错；本 Notebook 的 MusicGen runner 在检测到 MPS 时会自动改用 CPU。CUDA 可用时仍优先使用 CUDA。

不要直接 `pip install audiocraft`，它会拉取 `xformers<0.0.23`，并强制旧版 `av==11.0.0`。本 Notebook 用 `--no-deps` 安装 AudioCraft，再显式安装较新的 PyAV；这样通常会拿到可用 wheel，并避开 `av==11.0.0` 与 FFmpeg 8 头文件不兼容的问题。若必须复现 AudioCraft 原始依赖，请用 conda-forge 安装 `av=11` 与兼容 FFmpeg，或在本地准备 FFmpeg 6；不要用 Homebrew FFmpeg 8 直接源码编译 `av==11.0.0`。Apple Silicon 源码编译 PyAV 时若出现 x86_64/arm64 架构混用，再临时设置 `ARCHFLAGS="-arch arm64"`。

如果之前已经装到了 NumPy 2.x 或较新的 Transformers 4.x，在 `venv_ch08_audiocraft` 中执行：

```bash
python -m pip install --force-reinstall "numpy==1.26.4" "transformers==4.31.0"
```

Linux/CUDA 若需要 memory-efficient attention，可以按 AudioCraft/xformers 官方组合安装真实 `xformers`；CPU/MPS/普通 CUDA 推理路径不需要在本 Notebook 安装 `xformers`。本章的教学环境刻意使用 `av>=14` 和缺省 xformers shim，因此 `pip check` 仍会报告 `av`/`xformers` metadata 冲突；以 MusicGen import 检查和 Notebook 实际运行结果为准，不要为了消除这两条提示把 `av` 降回 11 或强装旧 `xformers`。


In [ ]:
from pathlib import Path
import os
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Audio, display

from _common.config import load_yaml_config
from _common.device_utils import choose_device
from _common.paths import portable_path
from evaluation.comparison_table import append_model_comparison
from model_runners.base import GenerationRequest
from model_runners.conditioning import build_conditioning_rows

OUTPUT_TABLES = ROOT / "outputs" / "tables"
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

def rel(path):
    return portable_path(path, ROOT)

def resolve_device(config=None):
    requested = os.getenv("CHAPTER08_DEVICE")
    if requested is None and config is not None:
        requested = str(config.get("device", "auto"))
    return choose_device(requested or "auto")

def print_setup_guidance(status):
    print(status.reason)
    if status.next_action:
        print(status.next_action)
    print("After completing the setup or moving to compatible hardware, rerun this Notebook; it will load and run the model directly.")

def print_runtime_guidance(error):
    print(str(error))
    print("Resolve the message above, then rerun this Notebook or the current cell.")

from model_runners.musicgen import MusicGenRunner

runner = MusicGenRunner()
status = runner.check_environment()
config = load_yaml_config(ROOT / "configs" / "musicgen_inference.yaml")
display(pd.DataFrame([status.as_row()]))


In [ ]:
display(pd.DataFrame(build_conditioning_rows("musicgen")))
display(pd.DataFrame(config.get("prompts", [])))
if config.get("audio_prompt"):
    display(pd.DataFrame([config["audio_prompt"]]))


In [ ]:
if status.available:
    prompt = config["prompts"][0]
    request = GenerationRequest(
        prompt=prompt["text"],
        prompt_id=prompt["prompt_id"],
        duration_sec=float(config.get("duration_seconds", 8)),
        output_dir=ROOT / config["outputs"]["audio_dir"],
        extra={
            "model_name": os.getenv("CHAPTER08_MUSICGEN_MODEL", config.get("model_name", runner.model_name)),
            "device": resolve_device(config),
        },
    )
    try:
        result = runner.generate(request)
    except RuntimeError as exc:
        print_runtime_guidance(exc)
    else:
        append_model_comparison(
            ROOT / config["outputs"]["table_csv"],
            {
                "model_name": result.model_name,
                "prompt_id": result.prompt_id,
                "dataset_context": "text prompt",
                "duration_sec": result.duration_sec,
                "wall_time_sec": result.wall_time_sec,
                "device": result.device,
                "output_audio_path": result.output_audio_path,
            },
        )
        display(Audio(str(result.output_audio_path)))
else:
    print_setup_guidance(status)


In [ ]:
audio_cfg = config.get("audio_prompt", {})
audio_prompt_path = ROOT / audio_cfg.get("path", "")
if status.available and audio_cfg and audio_prompt_path.exists():
    request = GenerationRequest(
        prompt=audio_cfg["text"],
        prompt_id=audio_cfg["prompt_id"],
        duration_sec=float(audio_cfg.get("duration_seconds", config.get("duration_seconds", 8))),
        output_dir=ROOT / config["outputs"]["audio_dir"],
        extra={
            "model_name": os.getenv("CHAPTER08_MUSICGEN_MODEL", config.get("model_name", runner.model_name)),
            "device": resolve_device(config),
            "prompt_duration_sec": float(audio_cfg.get("prompt_duration_sec", 2.0)),
        },
    )
    try:
        result = runner.generate_continuation(request, audio_prompt_path)
    except RuntimeError as exc:
        print_runtime_guidance(exc)
    else:
        append_model_comparison(
            ROOT / config["outputs"]["table_csv"],
            {
                "model_name": result.model_name,
                "prompt_id": result.prompt_id,
                "dataset_context": "author audio continuation prompt",
                "duration_sec": result.duration_sec,
                "wall_time_sec": result.wall_time_sec,
                "device": result.device,
                "output_audio_path": result.output_audio_path,
            },
        )
        display(Audio(str(result.output_audio_path)))
elif audio_cfg:
    print("Audio prompt not found:", rel(audio_prompt_path))
    print("Download the author audio pack to CODE/datasets/audio_author/chapter_08_author, then rerun this Notebook.")
